# Lead AI · Gouvernance des données

**Objectif :** Gestion et gouvernance des données, catalogues, RGPD et AI Act.

Lance ce notebook depuis le dossier `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../labs"))   # brewly, fakeapi
os.chdir("../data")                               # penguins.csv, titanic.csv, tips.csv

---
## Gouvernance et qualité des données

Un modèle n'est jamais meilleur que ses données. À l'échelle d'une entreprise, la question n'est plus « mon jeu de données est-il propre ? », mais « qui est responsable de quelles données, avec quelle qualité, qui y a accès, et qu'a-t-on le droit d'en faire ? ». C'est la **gouvernance des données**, et c'est le premier sujet d'un Lead AI.

## Les rôles

| Rôle | Responsabilité |
| --- | --- |
| **Propriétaire** (*data owner*) | Un responsable métier : il décide de l'usage et des accès |
| **Intendant** (*data steward*) | Il veille à la qualité, aux définitions, à la documentation |
| **Ingénieur de données** | Il construit et maintient les flux (ETL, entrepôt) |
| **DPO** | Le délégué à la protection des données : la conformité au RGPD |

## Les dimensions de la qualité

- **Complétude** : quelle part des valeurs est renseignée ?
- **Unicité** : y a-t-il des doublons ?
- **Validité** : les valeurs respectent-elles les règles (un âge entre 0 et 120, un code pays existant) ?
- **Cohérence** : la même information est-elle identique d'un système à l'autre ?
- **Fraîcheur** : les données sont-elles à jour ?

In [ ]:
import numpy as np
import pandas as pd
clients = pd.DataFrame({"id": [1, 2, 3, 3], "age": [34, 230, np.nan, np.nan], "pays": ["FR", "XX", "BE", "BE"]})
print("complétude :", clients.notna().mean().round(2).to_dict())
print("doublons   :", clients.duplicated().sum())
print("âges invalides :", (~clients["age"].dropna().between(0, 120)).sum())

## Les contrats de données

Un **contrat de données** est un accord écrit entre l'équipe qui produit une table et celles qui la consomment : colonnes, types, valeurs autorisées, fraîcheur, et qui prévenir en cas de changement. Il est vérifié automatiquement à chaque chargement, et un chargement non conforme est bloqué **avant** d'atteindre le modèle. Des outils comme **Great Expectations**, **Soda** ou **dbt tests** font ce travail.

## Les données personnelles

Le RGPD (leçon Réglementation) impose de savoir où sont les **données personnelles**, de limiter leur usage à une finalité, et de les protéger. En pratique :

- **Classer** chaque colonne : publique, interne, confidentielle, personnelle.
- **Minimiser** : ne collecter et ne garder que ce qui sert.
- **Pseudonymiser** ou **masquer** : remplacer un e-mail par un identifiant, retirer les numéros de téléphone des textes avant de les envoyer à un LLM.
- **Tracer les accès** : qui a lu quoi, et quand.

> **À retenir** : des rôles clairs (propriétaire, intendant), cinq dimensions de qualité mesurées automatiquement, des contrats entre producteurs et consommateurs, et un inventaire des données personnelles, masquées dès que possible.

### Exercice 1 · Un rapport de qualité

Écris `rapport_qualite(df, regles)` : `regles` associe un nom de colonne à une fonction qui renvoie True si **une** valeur est valide. Renvoie un dictionnaire avec `completude` (dictionnaire colonne → part de valeurs non nulles, arrondie à 3 décimales), `doublons` (nombre de lignes entièrement dupliquées) et `invalides` (dictionnaire colonne → nombre de valeurs **non nulles** qui ne respectent pas la règle, pour les colonnes de `regles`).

<details><summary>Indice</summary>

df[c].notna().mean() pour la complétude ; df.duplicated().sum() pour les doublons ; df[c].dropna().map(regle) pour tester chaque valeur.
</details>

In [ ]:
import numpy as np
import pandas as pd

clients = pd.DataFrame({
    "id": [1, 2, 3, 4, 4, 6, 7, 8],
    "email": ["ana@mail.fr", "bob@mail", None, "dee@mail.fr", "dee@mail.fr", "eli@mail.fr", None, "gus@mail.fr"],
    "age": [34, 17, 52, 41, 41, 230, 29, np.nan],
    "pays": ["FR", "FR", "BE", "XX", "XX", "FR", "DE", "FR"],
})

regles = {
    "age": lambda v: 0 <= v <= 120,
    "pays": lambda v: v in {"FR", "BE", "DE", "ES", "IT"},
    "email": lambda v: "@" in v and "." in v.split("@")[-1],
}

def rapport_qualite(df, regles):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
r = rapport_qualite(clients, regles)
assert isinstance(r, dict) and set(r) == {"completude", "doublons", "invalides"}, "Le rapport doit contenir completude, doublons et invalides."
assert r["completude"] == {"id": 1.0, "email": 0.75, "age": 0.875, "pays": 1.0}, f"Complétude attendue : id 1.0, email 0.75, age 0.875, pays 1.0 ; tu as {r['completude']}."
assert r["doublons"] == 1, "Il y a une ligne entièrement dupliquée (le client 4)."
assert r["invalides"] == {"age": 1, "pays": 2, "email": 1}, f"Invalides attendus : age 1 (230), pays 2 (XX deux fois), email 1 (bob@mail) ; tu as {r['invalides']}. Ne compte que les valeurs non nulles."
print("Bravo, tout est correct.")

### Exercice 2 · Masquer les données personnelles

Écris `masquer(texte)` qui remplace : les adresses e-mail par `[EMAIL]` ; les numéros de téléphone français par `[TEL]` (10 chiffres commençant par 0, ou `+33` suivi de 9 chiffres, éventuellement séparés par des espaces, des points ou des tirets) ; les IBAN français par `[IBAN]` (`FR` suivi de 25 caractères alphanumériques, éventuellement par groupes séparés d'espaces). Utilise le module `re`.

<details><summary>Indice</summary>

Trois expressions régulières. Pour le téléphone : (?:\+33[\s.-]?[1-9]|\b0[1-9]) suivi de 4 groupes (?:[\s.-]?\d{2}). Remplace l'IBAN avant le téléphone.
</details>

In [ ]:
import re

def masquer(texte):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
cas = {
    "Écris à jean.dupont+test@mail.example.com demain": "Écris à [EMAIL] demain",
    "Appelle le 06 12 34 56 78.": "Appelle le [TEL].",
    "ou le 01.23.45.67.89 ou le 0612345678": "ou le [TEL] ou le [TEL]",
    "Depuis l'étranger : +33 6 12 34 56 78": "Depuis l'étranger : [TEL]",
    "IBAN : FR76 3000 6000 0112 3456 7890 189": "IBAN : [IBAN]",
    "IBAN collé FR7630006000011234567890189 fin": "IBAN collé [IBAN] fin",
    "Commande 2024 de 3 articles, réf. 12345": "Commande 2024 de 3 articles, réf. 12345",
}
for entree, attendu in cas.items():
    obtenu = masquer(entree)
    assert obtenu == attendu, f"masquer({entree!r}) devrait donner {attendu!r} ; tu obtiens {obtenu!r}."
print("Bravo, tout est correct.")

### Exercice 3 · Vérifier un contrat de données

Écris `verifier_contrat(df, contrat)` qui renvoie la liste des violations, dans l'ordre des colonnes du contrat, avec exactement ces messages : `"<col> : colonne manquante"` (et rien d'autre pour cette colonne) ; `"<col> : <n> valeurs nulles"` si `nullable` vaut False et qu'il y a des nulls ; `"<col> : <n> valeurs hors de [<min>, <max>]"` si `min`/`max` sont donnés (sur les valeurs non nulles) ; `"<col> : <n> valeurs non autorisées"` si `valeurs` est donné (sur les valeurs non nulles).

<details><summary>Indice</summary>

Pour chaque colonne : d'abord la présence (continue si absente), puis les nulls, puis min/max avec between, puis valeurs avec isin, toujours sur df[col].dropna().
</details>

In [ ]:
import numpy as np
import pandas as pd

clients = pd.DataFrame({
    "id": [1, 2, 3, 4, 4, 6, 7, 8],
    "email": ["ana@mail.fr", "bob@mail", None, "dee@mail.fr", "dee@mail.fr", "eli@mail.fr", None, "gus@mail.fr"],
    "age": [34, 17, 52, 41, 41, 230, 29, np.nan],
    "pays": ["FR", "FR", "BE", "XX", "XX", "FR", "DE", "FR"],
})

contrat = {
    "id": {"nullable": False},
    "email": {"nullable": False},
    "age": {"nullable": True, "min": 18, "max": 120},
    "pays": {"nullable": False, "valeurs": ["FR", "BE", "DE"]},
    "date_inscription": {"nullable": False},
}

def verifier_contrat(df, contrat):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
r = verifier_contrat(clients, contrat)
attendu = ["email : 2 valeurs nulles", "age : 2 valeurs hors de [18, 120]", "pays : 2 valeurs non autorisées", "date_inscription : colonne manquante"]
assert r == attendu, f"Violations attendues :\n{attendu}\nTu obtiens :\n{r}"
assert verifier_contrat(clients, {"id": {"nullable": False}}) == [], "Un contrat respecté ne donne aucune violation."
print("Bravo, tout est correct.")

---
## Catalogue de données et lignage

Dans une entreprise, il y a des milliers de tables. Un analyste cherche « le chiffre d'affaires par client » : laquelle utiliser ? Qui la maintient ? Est-elle à jour ? Contient-elle des données personnelles ? Le **catalogue de données** répond à ces questions, comme un moteur de recherche interne pour les données.

## Ce que contient un catalogue

- Les **métadonnées techniques** : schéma, types, volumétrie, date de mise à jour, collectées automatiquement depuis les bases, l'entrepôt et les outils (dbt, Airflow, Tableau).
- Les **métadonnées métier** : description, glossaire (« client actif = au moins un achat sur 12 mois »), propriétaire, tags (`PII`, `certifié`).
- Le **lignage** (*lineage*) : d'où vient chaque table et ce qui en dépend, jusqu'aux tableaux de bord et aux modèles.
- La **qualité** : les résultats des contrôles de la leçon précédente.

Les outils open source les plus utilisés sont **OpenMetadata** (celui du programme) et **DataHub** ; côté cloud, AWS Glue Data Catalog, Google Dataplex ou Microsoft Purview.

## Pourquoi le lignage est si utile

```python
crm.clients ──┐
              ├──▶ entrepot.clients_propres ──▶ marts.ventes_par_client ──▶ dashboard.ca_mensuel
erp.commandes ┘                                         └──────────────────▶ modele.churn
```

- **Analyse d'impact** : avant de renommer une colonne de `crm.clients`, on voit immédiatement qu'un tableau de bord et un modèle en dépendent.
- **Analyse de cause** : un tableau de bord affiche un chiffre faux ? On remonte le lignage jusqu'à la source fautive.
- **Conformité** : un client demande la suppression de ses données (droit à l'effacement) ; le lignage montre toutes les tables où elles ont été copiées.

> **À retenir** : un catalogue = métadonnées techniques + métier + lignage + qualité. Le lignage est un graphe orienté : on le parcourt vers l'aval pour l'impact, vers l'amont pour la cause.

**À faire :**

- [ ] Lancer OpenMetadata avec Docker (guide « Quickstart ») et y connecter une base PostgreSQL ou un fichier CSV
- [ ] Ajouter une description, un propriétaire et un tag PII à une table
- [ ] Observer le lignage généré automatiquement à partir d'un projet dbt ou d'un DAG Airflow

### Exercice 1 · Parcourir le lignage

Le lignage est un dictionnaire : chaque actif pointe vers la liste des actifs construits à partir de lui. Écris `en_aval(lignage, actif)`, l'ensemble de tous les actifs qui dépendent de `actif`, directement ou indirectement, et `en_amont(lignage, actif)`, l'ensemble de toutes ses sources. Puis `impact(lignage, actif)` : la liste triée des actifs en aval dont le nom commence par `dashboard.` ou `modele.`.

<details><summary>Indice</summary>

Un parcours avec une pile et un ensemble « vus ». Pour remonter, construis le graphe inverse, puis réutilise en_aval.
</details>

In [ ]:
lignage = {
    "crm.clients": ["entrepot.clients_propres"],
    "erp.commandes": ["entrepot.commandes_propres"],
    "entrepot.clients_propres": ["marts.ventes_par_client", "marts.segments"],
    "entrepot.commandes_propres": ["marts.ventes_par_client"],
    "marts.ventes_par_client": ["dashboard.ca_mensuel", "modele.churn"],
    "marts.segments": ["dashboard.marketing"],
}

def en_aval(lignage, actif):
    pass

def en_amont(lignage, actif):
    pass

def impact(lignage, actif):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
assert en_aval(lignage, "erp.commandes") == {"entrepot.commandes_propres", "marts.ventes_par_client", "dashboard.ca_mensuel", "modele.churn"}, "en_aval doit suivre le lignage jusqu'au bout."
assert en_aval(lignage, "dashboard.ca_mensuel") == set(), "Un tableau de bord n'a rien en aval."
assert en_amont(lignage, "modele.churn") == {"marts.ventes_par_client", "entrepot.clients_propres", "entrepot.commandes_propres", "crm.clients", "erp.commandes"}, "en_amont doit remonter toutes les sources : inverse d'abord le graphe."
assert impact(lignage, "crm.clients") == ["dashboard.ca_mensuel", "dashboard.marketing", "modele.churn"], "impact : les tableaux de bord et modèles en aval, triés."
cycle = {"a": ["b"], "b": ["a"]}
assert en_aval(cycle, "a") == {"a", "b"}, "Avec un cycle, le parcours doit s'arrêter : garde un ensemble des actifs déjà vus."
print("Bravo, tout est correct.")

### Exercice 2 · Classer automatiquement les colonnes

Écris `classer(df)` qui renvoie un dictionnaire colonne → `"PII"` ou `"interne"`. Une colonne est `PII` si son nom (en minuscules) contient l'un des mots de `MOTS_PII`, **ou** si plus de la moitié de ses valeurs non nulles (converties en texte) ressemblent à un e-mail (`EMAIL.fullmatch`) ou à un numéro de téléphone (`TEL.fullmatch`).

<details><summary>Indice</summary>

Pour chaque colonne : any(m in c.lower() for m in MOTS_PII), puis la part des valeurs (dropna().astype(str)) qui passent EMAIL.fullmatch ou TEL.fullmatch.
</details>

In [ ]:
import re
import pandas as pd

MOTS_PII = ["nom", "prenom", "email", "mail", "tel", "adresse", "naissance", "iban"]
EMAIL = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
TEL = re.compile(r"(?:\+33|0)[1-9](?:[\s.-]?\d{2}){4}")

df = pd.DataFrame({
    "client_id": [1, 2, 3],
    "nom_complet": ["Ana Ruiz", "Bob Lee", "Cléo Martin"],
    "contact": ["ana@mail.fr", "bob@mail.fr", None],
    "commentaire": ["rappeler", "06 12 34 56 78", "ok"],
    "montant": [12.5, 40.0, 7.2],
    "joignable": ["0612345678", "07 11 22 33 44", "+33 6 98 76 54 32"],
})

def classer(df):
    pass

Exécute la cellule suivante pour vérifier ta réponse.

In [ ]:
r = classer(df)
attendu = {"client_id": "interne", "nom_complet": "PII", "contact": "PII", "commentaire": "interne", "montant": "interne", "joignable": "PII"}
assert r == attendu, f"Classement attendu : {attendu} ; tu as {r}."
assert classer(pd.DataFrame({"Email_Pro": [None, None]})) == {"Email_Pro": "PII"}, "Le nom de colonne se compare en minuscules."
print("Bravo, tout est correct.")